# Hackathon AITC: huấn luyện yolo26n-seg

Notebook này được mở thẳng từ nhánh `main` trên GitHub, nên luôn là bản mới nhất và không ai sửa được bản dùng chung. Mọi chỉnh sửa trong Colab chỉ nằm trong phiên của bạn. Muốn đổi notebook hoặc code thì mở PR vào `main`.

1. **Runtime → Change runtime type → GPU** (T4 là đủ).
2. Chạy lần lượt các ô. Ô 1 tải code mới nhất từ `main`.
3. Dữ liệu và kết quả nằm trên Google Drive, ở thư mục `DRIVE_DIR` bên dưới.

In [ ]:
# 1. Tải code mới nhất từ nhánh main và cài thư viện
import os
import subprocess
import sys
import tempfile

repo_url = 'https://github.com/kieuwangtruong/Hackathon_AITC.git'
repo_dir = tempfile.mkdtemp(prefix='hackathon_aitc_')
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', repo_url, repo_dir], check=True)
os.chdir(repo_dir)
sys.path.insert(0, repo_dir)
print('Code:', repo_dir)
subprocess.run(['git', 'log', '-1', '--format=%h %s'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

In [ ]:
# 2. Gắn Google Drive và chọn thư mục dữ liệu/kết quả
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Hackathon_AITC'  #@param {type:"string"}
DATASET_ZIP = 'datasets/du_lieu.zip'  #@param {type:"string"}
DATA_VERSION = 'v1'  #@param {type:"string"}
RUN_NAME = 'baseline'  #@param {type:"string"}
EPOCHS = 100  #@param {type:"integer"}
BATCH = 16  #@param {type:"integer"}
TEAM_ID = 48  # seed của đội do BTC cấp, không đổi

zip_path = os.path.join(DRIVE_DIR, DATASET_ZIP)
data_dir = f'/content/datasets/{DATA_VERSION}'
split_dir = f'/content/splits/{DATA_VERSION}'
runs_dir = os.path.join(DRIVE_DIR, 'runs')
sub_dir = os.path.join(DRIVE_DIR, 'submissions', RUN_NAME)
assert os.path.exists(zip_path), f'Không thấy {zip_path}'

In [ ]:
# 3. Giải nén, kiểm tra nhãn và chia train/val theo camera/địa điểm
!rm -rf {data_dir} {split_dir} && mkdir -p {data_dir}
!unzip -q {zip_path} -d {data_dir}
images = !find {data_dir} -type d -name images | head -1
root = os.path.dirname(images[0])
!python tools/check_labels.py --images {root}/images --labels {root}/labels
!python tools/split_dataset.py --images {root}/images --labels {root}/labels --out {split_dir}

In [ ]:
# 4. Huấn luyện (kết quả lưu trên Drive)
!python train.py --data {split_dir}/data.yaml --epochs {EPOCHS} --batch {BATCH} --seed {TEAM_ID} --project {runs_dir} --name {RUN_NAME}

In [ ]:
# 5. Xuất model.onnx, đóng zip nộp bài và chấm lại bản ONNX trên tập val
import glob
# Ultralytics đổi tên thành baseline2, baseline3... nếu RUN_NAME đã có; lấy lần chạy mới nhất.
best = max(glob.glob(f'{runs_dir}/{RUN_NAME}*/weights/best.pt'), key=os.path.getmtime)
print('Weights:', best)
!python export.py --weights {best} --out {sub_dir}
!python evaluate.py --model {sub_dir}/model.onnx --data {split_dir}/data.yaml
print('File nộp bài nằm trong', sub_dir)